# Post-processing and standard plots

This tutorial introduces the standardized post-processing interface. We run a small Vlasov–Ampère example, get its output as an autocomplete-friendly `Output`, and make the plots most commonly used to inspect a simulation.

For a production run you can skip the simulation setup and open its output folder with `struphy.open_output("path/to/sim")` instead.

In [ ]:
import os
import tempfile

from IPython.display import HTML

from struphy import (
    BinningPlot,
    BoundaryParameters,
    ButcherTableau,
    DerhamOptions,
    EnvironmentOptions,
    KernelDensityPlot,
    LoadingParameters,
    SavingParameters,
    Simulation,
    SortingParameters,
    Time,
    WeightsParameters,
    domains,
    equils,
    grids,
    maxwellians,
    perturbations,
)
from struphy.models import Maxwell, ViscousEulerSPH, VlasovAmpereOneSpecies

## Create a compact demonstration run

Post-processing operates on a completed run. The small setup below saves an electric field, a few marker trajectories, scalar diagnostics, and a binned $(\eta_1,v_1)$ distribution. These are the main output types handled by the plotting interface.

In [ ]:
def build_model():
    model = VlasovAmpereOneSpecies(alpha=1.0, epsilon=-1.0, with_B0=False)
    model.em_fields.e_field.save_data = True
    model.em_fields.phi.save_data = True
    model.kinetic_ions.var.save_data = True

    model.propagators.push_eta.options = model.propagators.push_eta.Options()
    model.propagators.coupling_va.options = model.propagators.coupling_va.Options()
    model.initial_poisson.options = model.initial_poisson.Options(stab_mat="M0")

    binplot = BinningPlot(
        slice="e1_v1",
        n_bins=(32, 32),
        ranges=((0.0, 1.0), (-5.0, 5.0)),
    )
    model.kinetic_ions.set_markers(
        loading_params=LoadingParameters(ppc=32, seed=1234),
        weights_params=WeightsParameters(control_variate=True),
        boundary_params=BoundaryParameters(),
        sorting_params=SortingParameters(boxes_per_dim=(4, 1, 1), do_sort=True),
        saving_params=SavingParameters(n_markers=12, binning_plots=(binplot,)),
    )

    background = maxwellians.Maxwellian3D(n=(1.0, None))
    model.kinetic_ions.var.add_background(background)
    density_mode = perturbations.ModesCos(ls=(1,), amps=(1e-3,))
    model.kinetic_ions.var.add_initial_condition(maxwellians.Maxwellian3D(n=(1.0, density_mode)))

    return model


model = build_model()

In [ ]:
demo_tmp = tempfile.TemporaryDirectory(prefix="struphy_postprocessing_")
demo_root = demo_tmp.name

env = EnvironmentOptions(
    out_folders=demo_root,
    sim_folder="vlasov_ampere_demo",
    save_restart=False,
)
sim = Simulation(
    model=model,
    env=env,
    time_opts=Time(dt=0.05, Tend=2.0),
    domain=domains.Cuboid(r1=2 * 3.141592653589793),
    equil=equils.HomogenSlab(),
    grid=grids.TensorProductGrid(num_elements=(16, 1, 1)),
    derham_opts=DerhamOptions(degree=(2, 1, 1)),
)
out = sim.run()
print(f"Raw output: {sim.env.path_out}")

## Process and load the output

`sim.run()` returns the run's output as a `Output`, which is also available later as `sim.output`. Scalars are read directly from the raw output; fields and particle products need post-processing, which runs with default options the first time they are accessed.

To choose options, call `out.process()` first. It evaluates saved FEEC fields and organizes particle diagnostics; `physical=True` additionally creates physical field components. Existing products made with the same options are reused, so re-running a cell is cheap.

Individual products are standard `xarray.DataArray` objects with named dimensions, coordinates, units, and labels. Arrays are loaded only when accessed. The simulation that produced them is `out.sim`.

In [ ]:
out.process(physical=True)

Products are arranged into clear namespaces. VS Code and interactive shells can complete the available names after a run is opened: fields are grouped by field species, while distribution and density products are grouped by species and saved slice. Every product can also be looked up by name, e.g. `out["electric_energy"]` or `out["kinetic_ions/e1_v1_density/f_binned"]`; flat catalogs remain available for code that needs to iterate over arbitrary products.

In [ ]:
print("scalars:", tuple(out.scalars.data_vars))
print("field species:", tuple(out.fields))
print("distribution species:", tuple(out.distributions))
print("particle species:", tuple(out.orbits))
print("all field products:", tuple(out.field_catalog))

phase_space = out.distributions.kinetic_ions.e1_v1_density.f_binned
print(phase_space)
print("dimensions:", phase_space.dims)
print("time coordinate:", phase_space.t)

## Scalar overview and time series

All standard plots are methods of `out.plot`, so no further imports are needed. They accept a product name or any array, and titles carry the run's numerical parameters.

`out.plot.scalars()` gives a quick overview of every recorded scalar. `out.plot.timeseries()` shows individual series on linear or logarithmic axes; `fit=(t0, t1)` adds an exponential fit restricted to that time window. Plots return an already-rendered `PlotResult`, which a notebook displays by itself; calling `.save()` never draws a second figure.

In [ ]:
out.plot.scalars()

In [ ]:
t_fit = 2.0 * out.sim.model.units.t  # in seconds, like every time coordinate of this run
energy_plot = out.plot.timeseries(
    "electric_energy",
    fit=(0.0, t_fit),
    fit_amplitude=True,
    title="Electric-field energy",
)
print("growth rate:", energy_plot.fit_results[0].rate)

# the same fit without a figure
print("growth rate:", out.analysis.growth_rate("electric_energy", window=(0.0, t_fit), amplitude=True).rate)

## Two-dimensional data

Choose the displayed dimensions with `x` and `y`, and pick one value for every other dimension by naming it: `t="last"` (or `"first"`), `t=-1` for a position, and `t=0.35` for the nearest coordinate value. Arrays can also be sliced beforehand with xarray's `.isel()` and `.sel()`. `coords="physical"` draws on the mapped coordinates instead of the logical ones.

In [ ]:
out.plot.slice(
    phase_space,
    x="e1",
    y="v1",
    t="last",
    equal_aspect=False,
    title="Final phase-space distribution",
)

For a compact view of the evolution, `out.plot.panels()` chooses evenly spaced snapshots in time. `shared_clim=True` makes panel colors directly comparable.

In [ ]:
out.plot.panels(
    phase_space,
    x="e1",
    y="v1",
    nrows=1,
    ncols=5,
    title="Phase-space evolution",
)

## Interactive plots

`out.plot.viewer()` adds one slider for every dimension not assigned to the display axes. In JupyterLab, run `%matplotlib widget` before this cell if `ipympl` is installed; the default inline backend still displays the initial frame. Keep the viewer alive so its callbacks remain connected. `out.plot.animation()` and `out.plot.frames()` sweep the same way.

In [ ]:
phase_viewer = out.plot.viewer(phase_space, x="e1", y="v1")
phase_viewer

Saved marker orbits are grouped by species. `out.plot.orbits()` draws their three-dimensional paths, while `max_markers` limits rendering cost for large production runs.

In [ ]:
out.plot.orbits("kinetic_ions", max_markers=12, show_paths=True)

`out.plot.animation()` and `out.plot.frames()` sweep the same data as the viewer. The animation is a Matplotlib `FuncAnimation`, displayed here as JavaScript; `frames()` writes one PNG per step and returns the paths.

In [ ]:
animation = out.plot.animation(phase_space, x="e1", y="v1", step=4)
HTML(animation.to_jshtml())

In [ ]:
frames = out.plot.frames(phase_space, "frames", x="e1", y="v1", step=10)
print("Wrote:", [os.path.basename(path) for path in frames])

For a run with a fluid equilibrium, `out.plot.equilibrium()` plots its radial profiles.

In [ ]:
out.plot.equilibrium()

## Derived quantities

`out.analysis` computes without drawing, and every result is an array that the plots accept. `drift()` subtracts the first sample, `relative_error()` gives the deviation relative to it, which is the usual way to inspect energy conservation.

In [ ]:
energy_error = out.analysis.relative_error("total_energy")
energy_drift = out.analysis.drift("total_energy")
print(f"largest drift of the total energy: {abs(energy_drift).max().item():.3e}")

out.plot.timeseries(energy_error, title="Conservation of the total energy")

`out.analysis.dispersion()` takes the space-time Fourier transform of a field along one direction and draws the spectrum. `slice_at` picks the direction of the transform (`None`) and the indices of the other two. Pass `disp_name` to overlay an analytic dispersion relation from `struphy.dispersion_relations.analytic`, and `fit_branches` to fit the dominant branches.

In [ ]:
omega, kvec, spectrum, _ = out.analysis.dispersion(
    "em_fields/e_field_log",
    slice_at=(None, 0, 0),
    do_plot=True,
)
print("spectrum:", spectrum.shape)

## Save standard output

Every `PlotResult` supports `.save(path)`. For a complete scalar report, `out.save_report()` writes a CSV table, an overview, and one PNG per scalar beneath `post_processing/report/`.

In [ ]:
written = out.save_report()
print("Wrote:")
for path in written:
    print(" ", os.path.relpath(path, out.path_out))

## Comparing runs

Every plot accepts arrays of other simulations, so comparing runs needs nothing special. Series are labelled by the run they come from, and the runs may have different time grids.

In [ ]:
sim_coarse = Simulation(
    model=build_model(),
    env=EnvironmentOptions(out_folders=demo_root, sim_folder="vlasov_ampere_coarse", save_restart=False),
    time_opts=Time(dt=0.1, Tend=2.0),
    domain=domains.Cuboid(r1=2 * 3.141592653589793),
    equil=equils.HomogenSlab(),
    grid=grids.TensorProductGrid(num_elements=(16, 1, 1)),
    derham_opts=DerhamOptions(degree=(2, 1, 1)),
)
out_coarse = sim_coarse.run()

out.plot.timeseries(
    out["electric_energy"],
    out_coarse["electric_energy"],
    title="Electric energy: dt = 0.05 against dt = 0.1",
)

## Other models

The interface is the same for every model; only the products differ. Two more short runs show the two product types the Vlasov–Ampère demo does not have: SPH densities, and vector fields on a mapped domain.

### SPH densities

A standing sound wave discretized with SPH markers. `KernelDensityPlot` reconstructs the density on a grid, which appears under `out.densities`, while `BinningPlot` produces the binned quantities under `out.distributions`.

In [ ]:
sph_model = ViscousEulerSPH(with_B0=False, with_viscosity=False)
sph_model.propagators.push_eta.options = sph_model.propagators.push_eta.Options(
    butcher=ButcherTableau(algo="forward_euler"),
)
sph_model.propagators.push_sph_p.options = sph_model.propagators.push_sph_p.Options(kernel_type="gaussian_1d")
sph_model.euler_fluid.set_markers(
    loading_params=LoadingParameters(ppb=8, loading="tesselation"),
    weights_params=WeightsParameters(),
    boundary_params=BoundaryParameters(),
    sorting_params=SortingParameters(boxes_per_dim=(12, 1, 1), dims_mask=(True, False, False)),
    saving_params=SavingParameters(
        binning_plots=(BinningPlot(slice="e1", n_bins=(32,), ranges=(0.0, 1.0)),),
        kernel_density_plots=(KernelDensityPlot(pts_e1=41, pts_e2=1),),
    ),
)
sph_model.euler_fluid.var.add_background(equils.ConstantVelocity())
sph_model.euler_fluid.var.add_perturbation(del_n=perturbations.ModesSin(ls=(1,), amps=(1.0e-2,)))

sph = Simulation(
    model=sph_model,
    env=EnvironmentOptions(out_folders=demo_root, sim_folder="sph_soundwave", save_restart=False),
    time_opts=Time(dt=0.03125, Tend=2.5, split_algo="Strang"),
    domain=domains.Cuboid(r1=2.5),
    grid=None,
    derham_opts=None,
)
out_sph = sph.run()

print("densities:", tuple(out_sph.density_catalog))
print("binned:", tuple(out_sph.distribution_catalog))

For a one-dimensional run, the clearest picture is a space-time map: the sweep dimension `t` may be used as a display axis.

In [ ]:
out_sph.plot.slice(
    "euler_fluid/view_0/n_sph",
    x="t",
    y="e1",
    e2=0,
    e3=0,
    title="SPH density of the sound wave",
)

Products are plain `xarray.DataArray` objects, so anything xarray can do works directly, for example profiles at selected times:

In [ ]:
density = out_sph["euler_fluid/view_0/n_sph"].isel(e2=0, e3=0)
density.isel(t=[0, len(density.t) // 4, len(density.t) // 2]).plot.line(x="e1")

### Vector fields on a mapped domain

A coaxial waveguide mode of the Maxwell model, on an annulus. With `physical=True` the post-processing also computes the Cartesian field components (`*_phy`), and `coords="physical"` draws them on the mapped grid, with the plane chosen by `plane`.

In [ ]:
a1, a2 = 2.326744, 3.686839

maxwell_model = Maxwell()
maxwell_model.propagators.maxwell.options = maxwell_model.propagators.maxwell.Options(algo="implicit")
maxwell_model.em_fields.e_field.add_perturbation(perturbations.CoaxialWaveguideElectric_r(m=3, a1=a1, a2=a2))
maxwell_model.em_fields.e_field.add_perturbation(perturbations.CoaxialWaveguideElectric_theta(m=3, a1=a1, a2=a2))
maxwell_model.em_fields.b_field.add_perturbation(perturbations.CoaxialWaveguideMagnetic(m=3, a1=a1, a2=a2))

coaxial = Simulation(
    model=maxwell_model,
    env=EnvironmentOptions(out_folders=demo_root, sim_folder="coaxial", save_restart=False),
    time_opts=Time(dt=0.05, Tend=2.0),
    domain=domains.HollowCylinder(a1=a1, a2=a2, Lz=2.0),
    equil=equils.HomogenSlab(),
    grid=grids.TensorProductGrid(num_elements=(24, 48, 1)),
    derham_opts=DerhamOptions(degree=(2, 2, 1), bcs=(("dirichlet", "dirichlet"), None, None)),
)
out_coaxial = coaxial.run()
out_coaxial.process(physical=True)

print("fields:", tuple(out_coaxial.field_catalog))
print("dimensions:", out_coaxial["em_fields/b_field_phy"].dims)

In [ ]:
out_coaxial.plot.slice(
    "em_fields/b_field_phy",
    x="e1",
    y="e2",
    t="last",
    component=2,
    e3=0,
    coords="physical",
    plane="XY",
    title="$B_z$ of the coaxial mode",
)

In [ ]:
out_coaxial.plot.panels(
    "em_fields/b_field_phy",
    x="e1",
    y="e2",
    component=2,
    e3=0,
    coords="physical",
    plane="XY",
    nrows=1,
    ncols=4,
    title="$B_z$ over time",
)

## Apply the workflow to another run

For an already completed simulation, possibly in a separate process without MPI, open its output folder:

```python
import struphy

out = struphy.open_output("/path/to/sim_1").process(physical=True)
out.sim.domain, out.sim.model.units  # the simulation, restored from disk without allocating
```

Use `out.scalars`, `out.fields`, `out.distributions`, `out.orbits`, and `out.densities`. Attribute access is the normal interactive API; the corresponding `*_catalog` mappings are intended for generic loops and tooling.